# EDA PLEGMA

## 1. Các trường dữ liệu trong dataset

Mục tiêu: kiểm kê **tên cột thực tế**, ý nghĩa, đơn vị và sự hiện diện theo house. Bước này chỉ đọc header, không tính thống kê dòng hay biến đổi dữ liệu.

In [1]:
import csv
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

DATASET = "PLEGMA"
DESCRIPTIONS = {
    "timestamp": ("Thời điểm đo", "datetime"),
    "V": ("Điện áp toàn nhà", "V"),
    "A": ("Dòng điện toàn nhà", "A"),
    "P_agg": ("Công suất tác dụng toàn nhà", "W"),
    "issues": ("Cờ tổng công suất thiết bị đo vượt P_agg", "0/1"),
    "internal_temperature": ("Nhiệt độ trong nhà", "°C"),
    "internal_humidity": ("Độ ẩm trong nhà", "%"),
    "external_temperature": ("Nhiệt độ ngoài trời", "°C"),
    "external_temparature": ("Nhiệt độ ngoài trời; lỗi chính tả tên cột trong file gốc", "°C"),
    "external_humidity": ("Độ ẩm ngoài trời", "%"),
    "appliance": ("Tên thiết bị", "chuỗi"),
    "wattage [W]": ("Công suất thiết bị trong metadata", "W"),
    "threshold [W]": ("Ngưỡng xác định trạng thái bật", "W"),
    "min_on (sec)": ("Thời gian bật tối thiểu", "giây"),
    "min_off (sec)": ("Thời gian tắt tối thiểu", "giây"),
}


root = Path.cwd()
if not (root / "data/cleaned").exists():
    root = root.parent.parent
base = root / "data/cleaned" / DATASET
present = defaultdict(set)
files = defaultdict(int)
for path in base.rglob("*.csv"):
    if path.name == "appliances_metadata.csv":
        group = "metadata thiết bị"
    elif "Environmental_data" in path.parts:
        group = "môi trường"
    else:
        group = "điện"
    house = next((part for part in path.parts if part.startswith("House_")), path.stem.replace("CLEAN_", ""))
    with path.open(encoding="utf-8-sig", newline="") as f:
        header = next(csv.reader(f))
    files[group] += 1
    for field in header:
        present[(group, field)].add(house)
rows = []
for (group, field), houses in sorted(present.items()):
    meaning, unit = DESCRIPTIONS.get(field, (f"Công suất tác dụng của thiết bị {field}" if group == "điện" else "Cần đối chiếu tài liệu nguồn", "W" if group == "điện" else ""))
    rows.append({"nhóm": group, "trường gốc": field, "ý nghĩa": meaning, "đơn vị": unit, "số house có trường": len(houses), "house": ", ".join(sorted(houses))})
dictionary = pd.DataFrame(rows)
print("Số file CSV theo nhóm:", dict(files))
display(dictionary.drop(columns="house"))

Số file CSV theo nhóm: {'môi trường': 118, 'điện': 120, 'metadata thiết bị': 13}


,nhóm,trường gốc,ý nghĩa,đơn vị,số house có trường
0,metadata thiết bị,appliance,Tên thiết bị,chuỗi,13
1,metadata thiết bị,min_off (sec),Thời gian tắt tối thiểu,giây,13
2,metadata thiết bị,min_on (sec),Thời gian bật tối thiểu,giây,13
3,metadata thiết bị,threshold [W],Ngưỡng xác định trạng thái bật,W,13
4,metadata thiết bị,wattage [W],Công suất thiết bị trong metadata,W,13
5,môi trường,external_humidity,Độ ẩm ngoài trời,%,13
6,môi trường,external_temparature,Nhiệt độ ngoài trời; lỗi chính tả tên cột tron...,°C,1
7,môi trường,external_temperature,Nhiệt độ ngoài trời,°C,13
8,môi trường,internal_humidity,Độ ẩm trong nhà,%,13
9,môi trường,internal_temperature,Nhiệt độ trong nhà,°C,13


### Các nhóm thông tin khác

Ngoài CSV điện và môi trường, mỗi house có `appliances_metadata.csv` (ngưỡng/công suất thiết bị) và một workbook XLSX khảo sát. XLSX là **biểu mẫu**, không phải bảng cột cố định; ba phần chính là thông tin dân cư (`SOCIODEMOGRAPHIC`), đặc điểm nhà (`BUILDING CHARACTERISTICS`) và thời gian sử dụng thiết bị (`TIME OF USE OF APPLIANCES`). Đoạn mã sau kiểm kê sheet để xác nhận đủ 13 workbook.

In [2]:
from openpyxl import load_workbook

workbooks = sorted(base.rglob("*.xlsx"))
workbook_info = []
for path in workbooks:
    wb = load_workbook(path, read_only=True, data_only=True)
    workbook_info.append({"house": next(part for part in path.parts if part.startswith("House_")), "sheet": ", ".join(wb.sheetnames), "số sheet": len(wb.sheetnames)})
    wb.close()
print("Số workbook khảo sát:", len(workbooks))
display(pd.DataFrame(workbook_info))

Số workbook khảo sát: 13


,house,sheet,số sheet
0,House_01,HOUSE 1,1
1,House_02,HOUSE 2,1
2,House_03,HOUSE 1,1
3,House_04,HOUSE 1,1
4,House_05,HOUSE 1,1
5,House_06,HOUSE 1,1
6,House_07,HOUSE 1,1
7,House_08,HOUSE 1,1
8,House_09,HOUSE 1,1
9,House_10,HOUSE 1,1


### Ghi chú diễn giải

- `V`/`A` và nhiều cột thiết bị chỉ xuất hiện ở một số house; xem bảng trên trước khi gộp dữ liệu.
- `external_temparature` là cách viết sai của `external_temperature` trong một phần file. Giữ nguyên tên gốc trong bước kiểm kê; chuẩn hóa sau khi chốt quy ước.
- `fridge`, `fridge_1`, `fridge_2` và `ac_1`–`ac_3` là các cảm biến/thiết bị khác nhau theo house.
- Nhịp danh định: điện 10 giây, môi trường 15 phút.

Nguồn mô tả: `data/raw/PLEGMA/PlegmaDataset_README_20240326.txt`.